# SSRO Experiment — Raw Waveform Readout (tprocv2)

This notebook performs Single-Shot ReadOut (SSRO) in two conditions:
- **|0⟩ preparation**: qubit initialised by wait (ground state), idle π-pulse replaced by identity, then readout
- **|1⟩ preparation**: qubit initialised by wait, then a π-pulse to flip to |1⟩, then readout

Readout is performed using **raw ADC buffer capture** (no DDC accumulation),
so the output is time-domain ADC samples that can be demodulated offline.

Each condition is repeated **1000 shots**.

Hardware assumed:
- Xilinx ZCU216 RFSoC with XM655 analog front-end
- QICK firmware with tprocv2
- Qubit drive on DAC channel `cfg['qubit_ch']`
- Readout drive on DAC channel `cfg['res_ch']`
- ADC on channel `cfg['adc_ch']`

## 1. Imports & QICK initialisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
from datetime import datetime

from qick import QickSoc
from qick.asm_v2 import AsmV2, QickParam
from qick.averager_program import NDAveragerProgram   # tprocv2 base class

# ── Connect to the board ──────────────────────────────────────────────────────
soc = QickSoc()
soccfg = soc
print(soccfg)

## 2. Experiment configuration

Edit the values in `cfg` to match your device (Dátil / Anacard resonator & qubit parameters).

In [ ]:
cfg = {
    # ── Channel mapping ───────────────────────────────────────────────────────
    "qubit_ch"      : 0,          # DAC channel driving the qubit
    "res_ch"        : 1,          # DAC channel driving the readout resonator
    "adc_ch"        : 0,          # ADC channel for readout

    # ── Qubit pulse (π-pulse) ─────────────────────────────────────────────────
    "qubit_freq"    : 4500.0,     # MHz  — qubit drive frequency
    "qubit_gain"    : 0.5,        # DAC full-scale fraction (0–1)
    "pi_sigma"      : 0.025,      # µs   — Gaussian σ for the π-pulse
    "pi_length"     : 0.200,      # µs   — total pulse length (≈ 4–5 σ)

    # ── Readout pulse ─────────────────────────────────────────────────────────
    "res_freq"      : 6505.1,     # MHz  — resonator drive frequency
    "res_gain"      : 0.3,        # DAC full-scale fraction
    "res_length"    : 2.0,        # µs   — readout pulse duration

    # ── Raw ADC capture ───────────────────────────────────────────────────────
    # Number of ADC samples to capture.  The ADC runs at ~4096 MHz (decimated
    # internally); after the 8× decimation fabric the effective rate seen by
    # the buffer is ~512 MSPS, so 1 µs ≈ 512 samples.
    # Capture a window long enough to cover the full readout pulse + ringdown.
    "raw_length"    : 1024,       # samples in the ADC buffer

    # ── Timing ────────────────────────────────────────────────────────────────
    "relax_delay"   : 500.0,      # µs   — wait between shots (T1 reset)

    # ── Statistics ───────────────────────────────────────────────────────────
    "shots"         : 1000,       # repetitions per state-prep condition
}

## 3. QICK program — raw-readout SSRO (tprocv2)

Two programs share an identical structure; the only difference is whether
a π-pulse on the qubit channel is played before the readout tone.

Key tprocv2 points:
- `declare_gen` / `declare_readout` set up channels at init time.
- `set_pulse_registers` / `pulse` schedule waveforms.
- `trigger` opens the ADC acquisition window.
- After `soc.run()`, raw samples are retrieved with `get_raw_samples()`.

In [ ]:
from qick.asm_v2 import AsmV2


class SSRORawProgram(AsmV2):
    """
    Single-Shot ReadOut program using raw ADC buffer capture.

    Parameters
    ----------
    soccfg : QickSoc
    cfg    : dict   — experiment configuration (see cell above)
    pi_pulse : bool — if True, play a π-pulse before readout (|1⟩ prep)
                      if False, skip the qubit drive         (|0⟩ prep)
    """

    def __init__(self, soccfg, cfg, pi_pulse: bool = False):
        self.cfg      = cfg
        self.pi_pulse = pi_pulse
        super().__init__(soccfg, reps=cfg["shots"])

    # ── channel / pulse declarations ────────────────────────────────────────
    def initialize(self):
        cfg = self.cfg

        # Qubit drive generator
        self.declare_gen(
            ch=cfg["qubit_ch"],
            nqz=1,
        )

        # Readout resonator drive generator
        self.declare_gen(
            ch=cfg["res_ch"],
            nqz=1,
        )

        # ADC readout channel — raw buffer mode
        # length = number of ADC buffer samples to capture
        self.declare_readout(
            ch=cfg["adc_ch"],
            length=cfg["raw_length"],
            freq=cfg["res_freq"],   # used to set DDC NCO, but we read raw
            gen_ch=cfg["res_ch"],
        )

        # ── π-pulse waveform (Gaussian envelope) ───────────────────────────
        # add_gauss returns the waveform name; play it only if pi_pulse=True
        self.pi_waveform = self.add_gauss(
            ch=cfg["qubit_ch"],
            name="pi_pulse",
            sigma=self.us2cycles(cfg["pi_sigma"], gen_ch=cfg["qubit_ch"]),
            length=self.us2cycles(cfg["pi_length"], gen_ch=cfg["qubit_ch"]),
        )

        # ── Readout tone (rectangular) ─────────────────────────────────────
        self.set_pulse_registers(
            ch=cfg["res_ch"],
            style="const",
            freq=cfg["res_freq"],
            phase=0,
            gain=cfg["res_gain"],
            length=self.us2cycles(cfg["res_length"], gen_ch=cfg["res_ch"]),
        )

        # ── Qubit π-pulse registers ────────────────────────────────────────
        self.set_pulse_registers(
            ch=cfg["qubit_ch"],
            style="arb",
            waveform="pi_pulse",
            freq=cfg["qubit_freq"],
            phase=0,
            gain=cfg["qubit_gain"],
        )

        self.synci(200)   # short guard after init

    # ── shot body (repeated `reps` times) ───────────────────────────────────
    def body(self):
        cfg = self.cfg

        # Optional π-pulse (|1⟩ prep)
        if self.pi_pulse:
            self.pulse(ch=cfg["qubit_ch"], t=0)
            # Wait for the π-pulse to finish before starting readout
            self.sync_all(self.us2cycles(cfg["pi_length"] + 0.05))

        # Readout tone + ADC trigger, simultaneous
        self.pulse(ch=cfg["res_ch"], t=0)
        self.trigger(
            adcs=[cfg["adc_ch"]],
            adc_trig_offset=self.us2cycles(0.05),   # small offset for cable delay
            t=0,
        )

        # Wait for the readout window to close before the next shot
        self.sync_all(self.us2cycles(cfg["res_length"] + 0.1))

        # Relax delay — let the qubit decay back to |0⟩ between shots
        self.sync_all(self.us2cycles(cfg["relax_delay"]))

## 4. Run both experiments

In [ ]:
# ── |0⟩ preparation (no π-pulse) ─────────────────────────────────────────────
print("Running |0⟩ prep experiment...")
prog0 = SSRORawProgram(soccfg, cfg, pi_pulse=False)

# acquire_decimated returns the raw buffer for every shot
# Shape: (shots, raw_length, 2) — axis-2 is [I_raw, Q_raw]
raw0 = prog0.acquire_decimated(
    soc,
    load_pulses=True,
    progress=True,
)
print(f"|0⟩ raw data shape: {raw0[0].shape}")

# ── |1⟩ preparation (π-pulse) ────────────────────────────────────────────────
print("\nRunning |1⟩ prep experiment...")
prog1 = SSRORawProgram(soccfg, cfg, pi_pulse=True)

raw1 = prog1.acquire_decimated(
    soc,
    load_pulses=True,
    progress=True,
)
print(f"|1⟩ raw data shape: {raw1[0].shape}")

## 5. Extract I/Q via software demodulation

The raw buffer contains the IF/RF signal at the ADC.  
We demodulate offline by multiplying with a reference cosine/sine at the
expected IF frequency and integrating over the readout window.

If your firmware routes the signal through the DDC before the buffer
(board-dependent), the raw samples may already be at baseband — in that
case skip the demodulation step and integrate directly.

In [ ]:
def demodulate(raw_shots: np.ndarray, fs_MHz: float, fIF_MHz: float) -> tuple:
    """
    Software IQ demodulation of raw ADC shots.

    Parameters
    ----------
    raw_shots : ndarray, shape (shots, N, 2)
        Raw ADC samples. raw_shots[..., 0] = I-raw, raw_shots[..., 1] = Q-raw.
        If the firmware delivers a real signal, use raw_shots[..., 0] only.
    fs_MHz    : float  — ADC effective sample rate after decimation (MHz)
    fIF_MHz   : float  — intermediate frequency to demodulate at (MHz)

    Returns
    -------
    I, Q : ndarray, shape (shots,) — integrated I and Q per shot
    """
    shots, N, _ = raw_shots.shape
    t = np.arange(N) / fs_MHz                           # time axis in µs
    cos_ref = np.cos(2 * np.pi * fIF_MHz * t)           # shape (N,)
    sin_ref = np.sin(2 * np.pi * fIF_MHz * t)

    # Use the first (I) channel from the ADC; multiply and integrate
    signal = raw_shots[:, :, 0].astype(float)            # (shots, N)
    I = np.mean(signal * cos_ref[np.newaxis, :], axis=1)
    Q = np.mean(signal * sin_ref[np.newaxis, :], axis=1)
    return I, Q


# ── Parameters for demodulation ───────────────────────────────────────────────
# Adjust fs_eff to the actual buffer sample rate reported by your firmware.
# A common value for ZCU216 after CIC decimation is 512 MHz or 1024 MHz.
fs_eff_MHz = 512.0          # effective ADC buffer sample rate  [MHz]

# The IF frequency depends on your LO/NCO setting.
# If the resonator is in NQZ1 and the LO is set to res_freq, IF ≈ 0;
# otherwise set fIF to the actual beat frequency visible in the spectrum.
fIF_MHz    = 0.0            # set to your actual IF  [MHz]

# raw_shots from acquire_decimated is a list of one array per ADC channel
I0, Q0 = demodulate(raw0[0], fs_eff_MHz, fIF_MHz)
I1, Q1 = demodulate(raw1[0], fs_eff_MHz, fIF_MHz)

print(f"|0⟩  I mean={I0.mean():.4f}  Q mean={Q0.mean():.4f}")
print(f"|1⟩  I mean={I1.mean():.4f}  Q mean={Q1.mean():.4f}")

## 6. IQ scatter plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(I0, Q0, s=8, alpha=0.5, label=r"$|0\rangle$ prep", color="steelblue")
ax.scatter(I1, Q1, s=8, alpha=0.5, label=r"$|1\rangle$ prep", color="tomato")

# Mark centroids
ax.plot(I0.mean(), Q0.mean(), "b^", ms=12, zorder=5)
ax.plot(I1.mean(), Q1.mean(), "r^", ms=12, zorder=5)

ax.set_xlabel("I (a.u.)", fontsize=13)
ax.set_ylabel("Q (a.u.)", fontsize=13)
ax.set_title("SSRO — IQ plane (raw demodulation)", fontsize=14)
ax.legend(fontsize=12)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/ssro_iq_scatter.png", dpi=150)
plt.show()

## 7. Inspect raw waveforms (single shot)

Plot the time-domain ADC trace for a representative shot from each state
to verify the readout pulse shape and check for ringdown or reflections.

In [ ]:
shot_idx = 0   # which shot to plot
N        = cfg["raw_length"]
t_us     = np.arange(N) / fs_eff_MHz

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

for ax, raw, label, color in zip(
    axes,
    [raw0[0], raw1[0]],
    [r"$|0\rangle$ prep", r"$|1\rangle$ prep"],
    ["steelblue", "tomato"],
):
    trace_I = raw[shot_idx, :, 0]
    trace_Q = raw[shot_idx, :, 1]
    ax.plot(t_us, trace_I, color=color,      lw=0.8, label="I")
    ax.plot(t_us, trace_Q, color=color, lw=0.8, alpha=0.5, ls="--", label="Q")
    ax.set_ylabel("ADC counts", fontsize=11)
    ax.set_title(f"{label} — shot {shot_idx}", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (µs)", fontsize=12)
plt.tight_layout()
plt.savefig("figures/ssro_raw_waveforms.png", dpi=150)
plt.show()

## 8. Histogram of a single quadrature

A quick 1D histogram along the optimal projection axis — useful for estimating
assignment fidelity before running a full LDA/classifier.

In [ ]:
# Project onto the axis connecting the two centroids
dI = I1.mean() - I0.mean()
dQ = Q1.mean() - Q0.mean()
norm = np.sqrt(dI**2 + dQ**2)

if norm < 1e-12:
    print("Centroids are coincident — check demodulation frequency.")
else:
    proj0 = (I0 * dI + Q0 * dQ) / norm
    proj1 = (I1 * dI + Q1 * dQ) / norm

    bins = np.linspace(
        min(proj0.min(), proj1.min()),
        max(proj0.max(), proj1.max()),
        60,
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(proj0, bins=bins, alpha=0.6, label=r"$|0\rangle$", color="steelblue", density=True)
    ax.hist(proj1, bins=bins, alpha=0.6, label=r"$|1\rangle$", color="tomato",    density=True)
    ax.set_xlabel("Projection onto |0⟩–|1⟩ axis (a.u.)", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title("Single-quadrature histogram", fontsize=13)
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("figures/ssro_histogram.png", dpi=150)
    plt.show()

## 9. Save data to HDF5

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"ssro_raw_{timestamp}.h5"

# ── Stack into CNN-compatible layout ─────────────────────────────────────────
# raw0[0] and raw1[0] each have shape (shots, T, 2).
# Stack along a new axis-0 to get (2, shots, T, 2):
#   axis 0 → state index  (0 = |0⟩, 1 = |1⟩)
#   axis 1 → shot index
#   axis 2 → time sample
#   axis 3 → [I, Q]
# The CNN loader reads:  raw = f['results'][:]  # (2, N_shots, T, 2)
results = np.stack([raw0[0], raw1[0]], axis=0)   # (2, shots, T, 2)
print(f"results shape: {results.shape}  "
      f"(states={results.shape[0]}, shots={results.shape[1]}, "
      f"T={results.shape[2]}, IQ={results.shape[3]})")

with h5py.File(fname, "w") as f:
    # ── Primary dataset — matches CNN loader key 'results' ────────────────
    f.create_dataset("results", data=results, compression="gzip")

    # ── Demodulated IQ (optional, for quick sanity checks) ────────────────
    iq = f.create_group("iq")
    iq.create_dataset("I0", data=I0)
    iq.create_dataset("Q0", data=Q0)
    iq.create_dataset("I1", data=I1)
    iq.create_dataset("Q1", data=Q1)

    # ── Metadata ──────────────────────────────────────────────────────────
    meta = f.create_group("config")
    for k, v in cfg.items():
        meta.attrs[k] = v
    meta.attrs["fs_eff_MHz"] = fs_eff_MHz
    meta.attrs["fIF_MHz"]    = fIF_MHz
    meta.attrs["timestamp"]  = timestamp
    # State labels stored as attributes for reference
    meta.attrs["state_labels"] = ["0 (|0> prep, no pi-pulse)",
                                   "1 (|1> prep, pi-pulse)"]

print(f"Data saved → {fname}")
print(f"Load in CNN notebook with:  raw = f['results'][:]")

---
## Notes

### `acquire_decimated` vs `acquire`

| Method | Returns | Use case |
|--------|---------|----------|
| `acquire()` | Accumulated IQ pair per shot | Standard SSRO, fast |
| `acquire_decimated()` | Full time-domain buffer per shot | Raw waveform analysis, custom demodulation |

`acquire_decimated` is the tprocv2 method that gives you the raw buffer.
The returned array shape is `(shots, raw_length, 2)` where the last axis
is `[I_buffer, Q_buffer]`.

### Demodulation frequency

- If your firmware's DDC NCO is set to `res_freq` exactly, the raw buffer
  is already at baseband (IF ≈ 0) and you can skip demodulation.
- If there is an NCO offset or you are using a mixer-based setup, set
  `fIF_MHz` to the beat frequency visible in `np.fft.rfft(raw[shot,:,0])`.

### Buffer depth limit

The ZCU216 ADC buffer is typically 16k–64k samples.  If `raw_length` ×
`shots` exceeds the buffer the firmware will wrap; keep `raw_length` small
enough that a single readout window fits comfortably.

### Qubit reset

`relax_delay = 500 µs` assumes a T1 in the range 50–100 µs (≈ 5×T1).
Adjust to ≥ 5×T1 for your device.